In [ ]:
# General flow of data pipeline: 
# Data collecting ---> data preprocessing 
# Data collecting: collect data from Hugging Face ---> select categories ---> invistigate
# Imports
import os
import sys
from huggingface_hub import login
from datasets import load_dataset
from datasets import concatenate_datasets
from tqdm import tqdm

In [2]:
# Login to Hugging Face
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
# Load the dataset from Hugging Face
# Limit to 15000 items per category, then combine all categories
selected_categories = [
    "All_Beauty",
    "Electronics",
    "Appliances",
    "Home_and_Kitchen",
    "Gift_Cards"
]
print(f"Loading {len(selected_categories)} categories (up to 15000 items each)...")

category_datasets = []
for category in tqdm(selected_categories):

    category_dataset = load_dataset(
        "McAuley-Lab/Amazon-Reviews-2023",
        f"raw_meta_{category}",
        split="full",
        trust_remote_code=True
    )
    category_dataset = category_dataset.select(
        range(min(15000, len(category_dataset)))
    )
    # Add category column to the dataset as they are sperated datasets
    category_label = category.replace("_", " ").lower()
    category_dataset = category_dataset.map(
        lambda batch: {"category": [category_label] * len(batch["title"])},
        batched=True
    )
    category_datasets.append(category_dataset)
    print(f"{category}: {len(category_dataset):,} items loaded")

dataset = concatenate_datasets(category_datasets)
print(f"\nCombined dataset: {len(dataset):,} items loaded")

Loading 5 categories (up to 15000 items each)...


 20%|██        | 1/5 [00:07<00:28,  7.10s/it]

All_Beauty: 15,000 items loaded


 40%|████      | 2/5 [00:22<00:35, 11.82s/it]

Electronics: 15,000 items loaded


...

 80%|████████  | 4/5 [00:42<00:11, 11.19s/it]

Home_and_Kitchen: 15,000 items loaded


Map:   0%|          | 0/1137 [00:00<?, ? examples/s]

100%|██████████| 5/5 [00:47<00:00,  9.56s/it]

Gift_Cards: 1,137 items loaded

Combined dataset: 61,137 items loaded


In [4]:
# Invistigate the dataset
# This is Hugging Face object
# Not all data points have the price
dataset[6]

{'main_category': 'All Beauty',
 'title': 'Stain Bonnet For Baby Bonnet Silk Sleep Cap For Toddler Child Shower Cap Teens Kids',
 'average_rating': 4.1,
 'rating_number': 50,
 'features': [],
 'description': [],
 'price': 'None',
 'images': {'hi_res': ['https://m.media-amazon.com/images/I/71jrogkcveL._SL1500_.jpg',
   'https://m.media-amazon.com/images/I/713Z24d6hVS._SL1200_.jpg',
   'https://m.media-amazon.com/images/I/817Nj6u5zML._SL1500_.jpg',
   'https://m.media-amazon.com/images/I/71eeT1V8nDL._SL1000_.jpg',
   'https://m.media-amazon.com/images/I/71aPPNO7hML._SL1000_.jpg'],
  'large': ['https://m.media-amazon.com/images/I/51s4A5hDsPL.jpg',
   'https://m.media-amazon.com/images/I/51mGQ66k8mS.jpg',
   'https://m.media-amazon.com/images/I/5122qYD4KNL.jpg',
   'https://m.media-amazon.com/images/I/61gq01uu39L.jpg',
   'https://m.media-amazon.com/images/I/61tR+Q7HguL.jpg'],
  'thumb': ['https://m.media-amazon.com/images/I/51s4A5hDsPL._SS40_.jpg',
   'https://m.media-amazon.com/images/I/

In [5]:
# What's the most expensive item?
max_price = 0
max_item = None
for datapoint in tqdm(dataset):
    try:
        price = float(datapoint["price"])
        if price > max_price:
            max_item = datapoint
            max_price = price
    except ValueError:
        pass
print(f"The most expensive item is {max_item['title']} and it costs {max_price:,.2f}")

100%|██████████| 61137/61137 [00:13<00:00, 4517.44it/s]

The most expensive item is FE 600mm F4 GM Super Telephoto Lens (SEL600F40GM) and it costs 12,998.00


In [7]:
# Push the raw collected dataset to Hugging Face
import truststore
truststore.inject_into_ssl()   
username = "omarshaarawy11"
items_data = f"{username}/best_deal_01_raw_data"
dataset.push_to_hub(items_data)
print(f"Uploaded {len(dataset):,} raw items to {items_data}")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/62 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


README.md: 0.00B [00:00, ?B/s]

c:\Users\EGShaaraOm\AppData\Local\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\EGShaaraOm\.cache\huggingface\hub\datasets--omarshaarawy11--best_deal_01_raw_data. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Uploaded 61,137 raw items to omarshaarawy11/best_deal_01_raw_data
